# Ternary Weight Quantization + KD on CIFAR-10 — Colab runner

Thin launcher. All logic lives in `src/` and `scripts/` in the repo; this notebook only
mounts Drive, pulls the code, and calls it. Nothing here is experiment logic, so nothing
here needs to be reviewed against the pre-registration.

**Run cells 1–5 top to bottom.** Cell 5 is the long one.

**When Colab disconnects** — and over 20–35 GPU-hours it will — just reopen this notebook
and run every cell again. Cell 5 skips runs that already finished and resumes a partial
run from its `resume.pt` (amendment A3). Re-running it is always safe.

## 1. Environment

In [ ]:
import subprocess, torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4/L4/A100 GPU."
print("gpu        ", torch.cuda.get_device_name(0))
print("torch      ", torch.__version__, "| cuda", torch.version.cuda)
print("tf32 matmul", torch.backends.cuda.matmul.allow_tf32,
      "| tf32 cudnn", torch.backends.cudnn.allow_tf32)
print()
print("Note: the pre-registration is fp32-only and forbids TF32. src/train.py forces")
print("both flags False at run start; the values above are just Colab defaults.")


## 2. Drive

`runs/` must live on Drive — it is the only thing that survives a disconnect. CIFAR-10
stays on local disk instead, where it re-downloads in well under a minute and reads far
faster than Drive.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

RUNS_DIR = '/content/drive/MyDrive/atdl_runs'   # persistent
DATA_DIR = '/content/data'                      # ephemeral, re-downloaded per session
os.makedirs(RUNS_DIR, exist_ok=True)
print(RUNS_DIR, '->', len(os.listdir(RUNS_DIR)), 'run dirs so far')

## 3. Code

Every manifest records the commit it ran under (§11), so the code is pulled from git
rather than pasted into cells. Set `REPO_URL` once.

In [ ]:
REPO_URL = "https://github.com/pbhatt0022/ternary-kd-cifar10.git"
REPO_DIR = "/content/atdl"

import os, subprocess
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("running at commit", commit)

## 4. Phase 0 and 1 — pre-flight, then arm E

Phase 0 must pass before anything trains (§5). Phase 1 fixes arm E's architecture from
architecture alone, before training, so results cannot influence it.

In [ ]:
!python -m pytest tests -q
!python scripts/verify.py --data-dir {DATA_DIR}

In [ ]:
!python scripts/select_arm_e.py --write
!git -C {REPO_DIR} diff --stat src/arm_e_constants.py

## 5. Phases 2–5 — training, evaluation, sweep

`run_all.py` owns the phase ordering: teacher and its gate, then the headline arms, then
the single test-set pass, then the temperature sweep. It is idempotent — finished runs are
skipped, partial runs resume. Re-run this cell after every disconnect.

In [ ]:
!python scripts/run_all.py --runs-dir {RUNS_DIR} --data-dir {DATA_DIR}

## 6. Tables

Reads only files under `runs/`. No reported number is computed by hand (§11).

In [ ]:
!python scripts/aggregate.py --runs-dir {RUNS_DIR} --out {RUNS_DIR}/report